<a href="https://colab.research.google.com/github/kili-technology/kili-python-sdk/blob/main/recipes/labeling_bot_queue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to plug a labeling bot into a project's queue

A model — an LLM, a detector, a rules engine — can work on a Kili project as one more member:
an account with the labeler role, served its assets the way the labeling interface serves them to a
person. It asks for its next asset, works on it while nobody else gets it, submits or skips it, and
moves on until its queue is empty. People can label the same project at the same time: nobody is
handed an asset someone else holds.

In this recipe we will:

- create a classification project and add a few documents,
- run a bot that labels its queue, skipping what it cannot classify,
- look at what it did.

The queue belongs to the domain client, `kili.client_domain.Kili`. In production, give the bot an
account of its own, a member of the project assigned to its labeling step; here, the account
running the recipe creates the project, and is on its steps.

## Setup

In [ ]:
%pip install kili

In [ ]:
from kili.client_domain import Kili

kili = Kili(
    # api_endpoint='https://cloud.kili-technology.com/api/label/v2/graphql',
    # the line above can be uncommented and changed if you are working with an on-premise version of Kili
)

## A project to label

In [ ]:
json_interface = {
    "jobs": {
        "CLASSIFICATION_JOB": {
            "mlTask": "CLASSIFICATION",
            "instruction": "Document type",
            "required": 1,
            "isChild": False,
            "content": {
                "input": "radio",
                "categories": {"INVOICE": {"name": "Invoice"}, "RECEIPT": {"name": "Receipt"}},
            },
        }
    }
}

project_id = kili.projects.create(
    title="Labeling bot recipe", input_type="TEXT", json_interface=json_interface
)["id"]

In [ ]:
kili.assets.create_text(
    project_id=project_id,
    content_array=[
        "INVOICE n°2026-0042 — ACME Corp — Total due: 1 200 €",
        "Receipt — Coffee shop — 2 espressos, 4,40 €",
        "Dear Sir or Madam, please find attached our latest catalogue",
    ],
    external_id_array=["invoice-42", "receipt-7", "letter-1"],
)

## The bot

`classify` stands for your model: it returns the document's category, or `None` when it cannot
tell. `iter_to_label` hands the bot its assets one after another, each locked to it while the loop's
body runs — however long that takes. The body submits a label, or skips the asset with a reason:
the skipped asset leaves the queue, with the reason shown in Kili, until someone unskips it. If the
body raises, the asset goes back to the queue at once and the exception stops the loop.

In [ ]:
def classify(text: str):
    """Your model: the document's category, or None when it cannot tell."""
    lowered = text.lower()
    if "invoice" in lowered:
        return "INVOICE"
    if "receipt" in lowered:
        return "RECEIPT"
    return None

In [ ]:
from kili.exceptions import QueueError

for asset in kili.queue.iter_to_label(project_id=project_id):
    category = classify(asset.read_content().decode("utf-8"))
    if category is None:
        try:
            asset.skip(reason="The bot could not tell what kind of document this is.")
        except QueueError:  # skipping not allowed on this project, or on this asset
            asset.release()  # back to the queue, for a person to label
    else:
        asset.submit(json_response={"CLASSIFICATION_JOB": {"categories": [{"name": category}]}})
    print(f"{asset.external_id}: {asset.outcome} after {asset.seconds_held:.1f} s")

## What the bot did

The labels are the bot's, with the time it spent on each asset, and the assets moved on in the
workflow exactly as after a submit in the labeling interface.

In [ ]:
kili.assets.list(
    project_id=project_id,
    fields=["externalId", "currentStep.name", "currentStep.status", "latestLabel.secondsToLabel"],
)

## Cleanup

In [ ]:
kili.projects.delete(project_id=project_id)

## Conclusion

In this recipe, a bot labeled a project's queue as a teammate would: served the assets the labeling
interface would serve it, holding each while it worked, submitting or skipping it, and stopping when
the queue was empty. `iter_to_review` does the same for a review step, with `approve()` and
`send_back()`; an account that cannot work on the queue — not on its steps, or the wrong role — is
told why by a `QueueError`. The SDK reference's Queue page documents every method.